<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/notebooks/05_baseline_popularity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**chargement des données et du dev fixé**

In [1]:
import os, time, pickle
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")

NPY_DIR = "/content/drive/MyDrive/OTTO_Recommender_System/data/processed"
N, N_SESSIONS = 216_716_096, 12_899_779

aid_arr   = np.load(f"{NPY_DIR}/aid.npy")
ts_arr    = np.load(f"{NPY_DIR}/ts.npy")
type_arr  = np.load(f"{NPY_DIR}/type.npy")
sizes_arr = np.load(f"{NPY_DIR}/sizes.npy")
assert len(aid_arr) == N and sizes_arr.sum() == N

with open(f"{NPY_DIR}/dev_validation.pkl", "rb") as f:
    dev = pickle.load(f)
T, GT, prefix_len = dev["T"], dev["GT"], dev["prefix_len"]
cum = np.concatenate([[0], np.cumsum(sizes_arr, dtype=np.int64)])

WEIGHTS = {"clicks": 0.10, "carts": 0.30, "orders": 0.60}
TYPE2ID = {"clicks": 0, "carts": 1, "orders": 2}
print("Dev :", len(GT), "sessions | T =", pd.to_datetime(T, unit="s"))

Mounted at /content/drive
Dev : 300000 sessions | T = 2022-08-21 22:00:00


**tables de popularité (uniquement avant T)**

In [2]:
before = ts_arr < T
assert ts_arr[before].max() < T                      # garde-fou anti-fuite
print(f"Événements d'apprentissage : {before.sum():,}")

n_items = int(aid_arr.max()) + 1

def top_k(counts, k=40):
    return np.argsort(-counts, kind="stable")[:k].tolist()   # stable : ordre déterministe

cnt_all = np.bincount(aid_arr[before], minlength=n_items)
cnt_type = {t: np.bincount(aid_arr[before & (type_arr == i)], minlength=n_items)
            for t, i in TYPE2ID.items()}

lastw = before & (ts_arr >= T - 7 * 86400)           # semaine juste avant T
print(f"Événements de la dernière semaine avant T : {lastw.sum():,}")
cnt_type_w = {t: np.bincount(aid_arr[lastw & (type_arr == i)], minlength=n_items)
              for t, i in TYPE2ID.items()}

tops = {"B0": {t: top_k(cnt_all) for t in WEIGHTS},
        "B1": {t: top_k(cnt_type[t]) for t in WEIGHTS},
        "B2": {t: top_k(cnt_type_w[t]) for t in WEIGHTS}}

for t in WEIGHTS:
    print(t, "top-5 B1 :", tops["B1"][t][:5], "| top-5 B2 :", tops["B2"][t][:5])

Événements d'apprentissage : 163,955,181
Événements de la dernière semaine avant T : 55,407,112
clicks top-5 B1 : [1460571, 108125, 29735, 1733943, 832192] | top-5 B2 : [959208, 1603001, 108125, 1460571, 1164334]
carts top-5 B1 : [166037, 152547, 1733943, 231487, 29735] | top-5 B2 : [166037, 1603001, 1108416, 1022566, 1733943]
orders top-5 B1 : [231487, 166037, 1733943, 1629608, 756588] | top-5 B2 : [1603001, 166037, 231487, 1733943, 1022566]


**évaluation et bootstrap**

In [3]:
def per_session(preds, GT, k=20):
    """Hits et dénominateurs par session et par type (pour bootstrap)."""
    sids = list(GT)
    H = {t: np.zeros(len(sids)) for t in WEIGHTS}
    D = {t: np.zeros(len(sids)) for t in WEIGHTS}
    for i, sid in enumerate(sids):
        for t in WEIGHTS:
            gt = GT[sid][t]
            if gt:
                H[t][i] = len(set(preds[t].get(sid, [])[:k]) & gt)
                D[t][i] = min(k, len(gt))
    return H, D

def score_from(H, D, idx=None):
    out = {}
    for t in WEIGHTS:
        h = H[t] if idx is None else H[t][idx]
        d = D[t] if idx is None else D[t][idx]
        out[t] = h.sum() / d.sum()
    out["score"] = sum(WEIGHTS[t] * out[t] for t in WEIGHTS)
    return out

def bootstrap_ci(H, D, B=100, seed=0):
    r = np.random.default_rng(seed); n = len(next(iter(D.values())))
    s = [score_from(H, D, r.integers(0, n, n))["score"] for _ in range(B)]
    return np.percentile(s, [2.5, 97.5])

def boot_diff(HA, HB, D, B=100, seed=0):
    """IC95 de score(B) - score(A), mêmes sessions tirées pour A et B."""
    r = np.random.default_rng(seed); n = len(next(iter(D.values())))
    d = []
    for _ in range(B):
        idx = r.integers(0, n, n)
        d.append(score_from(HB, D, idx)["score"] - score_from(HA, D, idx)["score"])
    return np.percentile(d, [2.5, 97.5])

def const_preds(top):  # même liste pour toutes les sessions
    lists = {t: top[t][:20] for t in WEIGHTS}
    return {t: {sid: lists[t] for sid in GT} for t in WEIGHTS}

**résultats B0, B1, B2**

In [4]:
rows, HH = [], {}
for name in ["B0", "B1", "B2"]:
    H, D = per_session(const_preds(tops[name]), GT)
    HH[name] = H
    r = score_from(H, D); lo, hi = bootstrap_ci(H, D)
    rows.append({"modèle": name, **{k: round(v, 4) for k, v in r.items()},
                 "IC95 score": f"[{lo:.4f}, {hi:.4f}]"})
display(pd.DataFrame(rows))

for a, b in [("B0", "B1"), ("B1", "B2")]:
    lo, hi = boot_diff(HH[a], HH[b], D)
    print(f"Différence de score {b} - {a} : IC95 [{lo:+.4f}, {hi:+.4f}]")

,modèle,clicks,carts,orders,score,IC95 score
0,B0,0.0082,0.0062,0.0047,0.0055,"[0.0051, 0.0059]"
1,B1,0.0082,0.0078,0.0060,0.0067,"[0.0063, 0.0072]"
2,B2,0.0075,0.0068,0.0069,0.0070,"[0.0065, 0.0074]"


Différence de score B1 - B0 : IC95 [+0.0008, +0.0016]
Différence de score B2 - B1 : IC95 [-0.0001, +0.0005]


**B3, articles de la session + popularité par type**

In [5]:
def session_plus_pop(top):
    preds = {t: {} for t in WEIGHTS}
    for sid, k in prefix_len.items():
        a = cum[sid]
        seen = list(dict.fromkeys(map(int, aid_arr[a:a + k][::-1])))[:20]   # récents d'abord, sans doublon
        sset = set(seen)
        for t in WEIGHTS:
            fill = [x for x in top[t] if x not in sset]
            preds[t][sid] = (seen + fill)[:20]
    return preds

p3 = session_plus_pop(tops["B1"])
assert all(len(v) == 20 and len(set(v)) == 20 for t in p3 for v in p3[t].values())

H3, D = per_session(p3, GT)
r3 = score_from(H3, D); lo, hi = bootstrap_ci(H3, D)
print("B3 :", {k: round(v, 4) for k, v in r3.items()}, f"IC95 [{lo:.4f}, {hi:.4f}]")
lo, hi = boot_diff(HH["B1"], H3, D)
print(f"Différence B3 - B1 : IC95 [{lo:+.4f}, {hi:+.4f}]")

B3 : {'clicks': np.float64(0.3275), 'carts': np.float64(0.3121), 'orders': np.float64(0.5738), 'score': np.float64(0.4706)} IC95 [0.4667, 0.4761]
Différence B3 - B1 : IC95 [+0.4598, +0.4693]


**Derniere Analyse de cette phase 5**

In [6]:
sids = list(GT)
pl = np.array([prefix_len[s] for s in sids])
H3, D3 = per_session(p3, GT)

print("Recall de B3 par longueur de préfixe (nb de sessions entre parenthèses)")
rows = []
for lo, hi, lab in [(1,1,"1"), (2,2,"2"), (3,5,"3-5"), (6,10,"6-10"), (11,10**9,"11+")]:
    m = (pl >= lo) & (pl <= hi)
    row = {"préfixe": lab, "sessions": int(m.sum())}
    for t in WEIGHTS:
        d = D3[t][m].sum()
        row[t] = round(H3[t][m].sum() / d, 3) if d > 0 else np.nan
    rows.append(row)
display(pd.DataFrame(rows))

# Plafond session-only : on classe parfaitement les articles du préfixe
hits = {t: 0 for t in WEIGHTS}; den = {t: 0 for t in WEIGHTS}
for sid in sids:
    a, k = cum[sid], prefix_len[sid]
    ps = set(map(int, aid_arr[a:a + k]))
    for t in WEIGHTS:
        gt = GT[sid][t]
        if gt:
            hits[t] += min(len(gt & ps), 20)
            den[t]  += min(20, len(gt))
ceil = {t: hits[t] / den[t] for t in WEIGHTS}
print("Plafond session-only :", {t: round(v, 4) for t, v in ceil.items()},
      "| score :", round(sum(WEIGHTS[t] * ceil[t] for t in WEIGHTS), 4))

Recall de B3 par longueur de préfixe (nb de sessions entre parenthèses)


,préfixe,sessions,clicks,carts,orders
0,1,133658,0.273,0.371,0.360
1,2,52436,0.359,0.375,0.504
2,3-5,59060,0.366,0.337,0.572
3,6-10,28948,0.387,0.295,0.604
4,11+,25898,0.401,0.246,0.621


Plafond session-only : {'clicks': 0.3239, 'carts': 0.3139, 'orders': 0.5959} | score : 0.4841
